# SalesPath Colab Training

This notebook installs dependencies, runs a local environment server, validates rollout, and launches curriculum training.

In [ ]:
!pip install -U pip
!pip install fastapi uvicorn pydantic httpx torch transformers trl unsloth openenv

In [ ]:
# If the repo is not already present, clone it.
# !git clone https://github.com/<your-org-or-user>/salespath_env.git
# %cd salespath_env

%cd /content/salespath_env

In [ ]:
# Start the OpenEnv-compatible server in background.
!nohup python -m uvicorn salespath_env.server.app:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1 &
!sleep 3
!python -c "import httpx; r=httpx.get('http://127.0.0.1:8000/health', timeout=30); print(r.status_code, r.text)"

In [ ]:
# Rollout smoke test (single episode)
!python -m training.test_rollout

In [ ]:
# Curriculum run (example)
!python -m training.grpo_train --steps 30 --env-url http://127.0.0.1:8000 --model-name Qwen/Qwen2.5-0.5B-Instruct

## Optional: Push merged model to Hugging Face

Set your token first:

```python
import os
os.environ['HF_TOKEN'] = 'hf_xxx'
```

Then run:

```bash
python -m training.grpo_train --steps 100 --push-merged --hub-repo Imsachin010/salespath-qwen25-7b
```